# KRISHI-DRISHTI v4 round-3 training (Colab, Suhani account)
1. GPU check 2. fetch PlantDoc + PV via kagglehub 3. clone repo + patch manifest 4. train 5. export artifacts

In [ ]:
# 1. GPU check
!nvidia-smi
import tensorflow as tf
print('TF', tf.__version__)
print('GPUs', tf.config.list_physical_devices('GPU'))

In [ ]:
# 2. data: PlantDoc stage (public dataset) + PV corpus (public dataset) via kagglehub
import os
os.environ['KAGGLEHUB_CACHE'] = '/content/kagglehub_cache'
import kagglehub
from pathlib import Path
pd_base = Path(kagglehub.dataset_download('suhanikumari0104/krishi-plantdoc-stage'))
pd_root = next(c for c in pd_base.rglob('PlantDoc_Solanaceae') if c.is_dir() and any(c.rglob('*.jpg')))
print('PlantDoc root:', pd_root, len(list(pd_root.rglob('*.jpg'))), 'jpgs')
pv_base = Path(kagglehub.dataset_download('vipoooool/new-plant-diseases-dataset'))
pv_train = next(c for c in pv_base.rglob('train') if c.is_dir() and (c / 'Tomato___Late_blight').exists())
print('PV train dir:', pv_train)
os.environ['KRISHI_PV_TRAIN_DIR'] = str(pv_train)

In [ ]:
# 3. clone repo + patch holdout manifest paths for this machine
import subprocess, sys
REPO = '/content/KRISHI-DRISHTI'
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/gautamkr0104/KRISHI-DRISHTI.git', REPO], check=True)
man = Path(REPO) / 'ai_engine' / 'PLANTDOC_HOLDOUT_MANIFEST.json'
d = man.read_text(encoding='utf-8')
winprefix = 'C:' + '\\' * 2 + 'Users' + '\\' * 2 + 'DELL' + '\\' * 2 + '.cache' + '\\' * 2 + 'PlantDoc_Solanaceae'
d = d.replace(winprefix, str(pd_root))
d = d.replace('\\' * 2, '/')
man.write_text(d, encoding='utf-8')
print('manifest staged refs:', d.count(str(pd_root)))

In [ ]:
# 4. run v4 training (~1.5-2h on T4)
import subprocess, sys, os
r = subprocess.run([sys.executable, 'ai_engine/train_model_v4.py'], cwd='/content/KRISHI-DRISHTI',
                   env={**os.environ})
print('train exit code:', r.returncode)
assert r.returncode == 0, 'training failed - see log above'

In [ ]:
# 5. export artifacts for download
from pathlib import Path
import shutil
out = Path('/content/artifacts'); out.mkdir(exist_ok=True)
for f in ['krishi_drishti_model.tflite', 'krishi_drishti_base.keras', 'TRAINING_RUN_MANIFEST.json', 'ROUND2_FINETUNE_MANIFEST.json']:
    p = Path('/content/KRISHI-DRISHTI/ai_engine') / f
    if p.exists():
        shutil.copy2(p, out / f)
        print('exported:', f, p.stat().st_size, 'bytes')
print('DONE - download files from /content/artifacts')